# Workspace RayCluster + job client — distributed compute

Same pattern as Topic 1: attach or create the shared **`ray-workshop`** cluster, then `cluster.job_client`.

Runs `distributed_stats.py` with `@ray.remote` tasks across workers.

Leave the cluster up for Topic 3 (tear down at the end of Topic 3).


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from workshop_bootstrap import setup_workshop_path

REPO_ROOT = setup_workshop_path()
print(f"Repo root: {REPO_ROOT}")

## Authenticate

Same credentials as Topic 1 — OpenShift Console → **Copy login command** → Display token.


In [ ]:
from kube_authkit import AuthConfig, get_k8s_client
from codeflare_sdk import set_api_client, list_local_queues

OPENSHIFT_SERVER = "https://api.YOUR_CLUSTER:6443"
OPENSHIFT_TOKEN = "REPLACE_WITH_YOUR_TOKEN"

auth_config = AuthConfig(
    method="openshift",
    k8s_api_host=OPENSHIFT_SERVER.strip(),
    token=OPENSHIFT_TOKEN.strip(),
    verify_ssl=False,
)
set_api_client(get_k8s_client(config=auth_config))

NAMESPACE = "ray-workshop"
LOCAL_QUEUE = "ray-workshop-queue"

list_local_queues(NAMESPACE)

In [ ]:
from workshop_bootstrap import ensure_workshop_cluster

cluster = ensure_workshop_cluster(namespace=NAMESPACE, local_queue=LOCAL_QUEUE)
cluster.details()


In [ ]:
import time

client = cluster.job_client

submission_id = client.submit_job(
    entrypoint="python extras/scripts/distributed_stats.py",
    runtime_env={
        "working_dir": str(REPO_ROOT),
        "env_vars": {
            "INPUT_PATH": "extras/data/iris.csv",
            "NUM_PARTITIONS": "4",
        },
    },
)
print(f"Submitted: {submission_id}")

terminal = {"SUCCEEDED", "FAILED", "STOPPED"}
deadline = time.time() + 900

while time.time() < deadline:
    status = client.get_job_status(submission_id)
    print(f"Job {submission_id}: {status}")
    if str(status).split(".")[-1].upper() in terminal:
        break
    time.sleep(15)
else:
    raise TimeoutError(f"Timed out waiting for job {submission_id}")

print(client.get_job_logs(submission_id))

In [ ]:
# Optional — skip to keep the cluster for Topic 3
# cluster.down()
# print("Cluster down.")
print("Leaving cluster up for Topic 3. Uncomment cluster.down() above if needed.")


## Check results

Expect JSON partition summaries and `Aggregated row count: 30` in the job logs above.

Optional CLI (while the cluster is up):

```sh
oc logs -n ray-workshop -l ray.io/cluster=ray-workshop,ray.io/node-type=head -c ray-head --tail=100
```
